# PsyDefDetect — Defense Mechanism Classifier Training

**Pixelated Empathy** · DMRS-based defense mechanism detection

Fine-tunes `microsoft/deberta-v3-base` on the PSYDEFCONV dataset
with Focal Loss, R-Drop regularization, and 5-fold GroupKFold CV.

---

### Setup
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `train.json` and `test.json` when prompted
3. Run all cells

In [ ]:
# Cell 1: Install dependencies
%pip install -q transformers accelerate sentencepiece scikit-learn

In [ ]:
# Cell 2: Download dataset files automatically
import os
import urllib.request

os.makedirs('data', exist_ok=True)

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/ShenXiexs/Sam_PsyDefDetect-BioNLP2026/main/input_data/train.json',
    'data/train.json'
)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/ShenXiexs/Sam_PsyDefDetect-BioNLP2026/main/input_data/test.json',
    'data/test.json'
)

print('Datasets downloaded successfully to data/')

In [ ]:
# Cell 3: Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 4: Constants

DEFENSE_LABELS = {
    0: 'Neutral',
    1: 'Action Defenses',
    2: 'Major Image-Distorting',
    3: 'Disavowal',
    4: 'Minor Image-Distorting',
    5: 'Neurotic',
    6: 'Obsessional',
    7: 'High-Adaptive',
    8: 'Needs More Info',
}

DEFENSE_MATURITY = {
    0: None, 1: 0.0, 2: 0.14, 3: 0.29,
    4: 0.43, 5: 0.57, 6: 0.71, 7: 1.0, 8: None,
}

NUM_LABELS = 9
print('Constants defined')

In [ ]:
# Cell 5: Dataset loader

import json
import logging
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
from sklearn.model_selection import GroupKFold
from torch.utils.data import Dataset
from transformers import PreTrainedTokenizer

logger = logging.getLogger(__name__)

SPEAKER_MAP = {
    'seeker': 'Seeker', 'supporter': 'Supporter',
    'help-seeker': 'Seeker', 'helper': 'Supporter',
    'patient': 'Seeker', 'therapist': 'Supporter',
    'user': 'Seeker', 'system': 'Supporter',
}


@dataclass
class DialogueSample:
    sample_id: str
    dialogue_id: str
    turns: list = field(default_factory=list)
    target_text: str = ''
    target_turn_index: int = -1
    label: Optional[int] = None


def normalize_speaker(speaker):
    return SPEAKER_MAP.get(speaker.strip().lower().rstrip(':'), 'Seeker')


def format_dialogue(turns, target_text, max_turns=40, target_turn_index=-1):
    if len(turns) > max_turns:
        offset = len(turns) - max_turns
        turns = turns[offset:]
        if target_turn_index >= 0:
            target_turn_index -= offset

    parts = []
    target_norm = re.sub(r'\\s+', ' ', target_text.strip().lower())

    for i, turn in enumerate(turns):
        speaker = normalize_speaker(turn.get('speaker', 'Seeker'))
        text = turn.get('text', turn.get('content', '')).strip()
        is_target = (i == target_turn_index) if target_turn_index >= 0 else (
            re.sub(r'\\s+', ' ', text.strip().lower()) == target_norm
        )
        if is_target:
            parts.append(f'{speaker}: <t>{text}</t>')
        else:
            parts.append(f'{speaker}: {text}')

    return ' '.join(parts)


def compute_class_weights(labels):
    counts = np.zeros(NUM_LABELS, dtype=np.float64)
    for label in labels:
        if 0 <= label < NUM_LABELS:
            counts[label] += 1
    counts = np.maximum(counts, 1.0)
    weights = 1.0 / np.sqrt(counts)
    weights = weights * NUM_LABELS / weights.sum()
    return torch.tensor(weights, dtype=torch.float32)


def load_psydefconv(data_path, has_labels=True):
    with open(data_path, encoding='utf-8') as f:
        raw_data = json.load(f)
    if isinstance(raw_data, dict):
        raw_data = raw_data.get('data', raw_data.get('samples', []))

    samples = []
    for i, item in enumerate(raw_data):
        sample_id = str(item.get('id', f's_{i}'))
        dialogue_id = str(item.get('dialogue_id', sample_id.split('_')[0]))
        raw_dialogue = item.get('dialogue', item.get('conversation', []))
        turns = []
        if isinstance(raw_dialogue, list):
            for turn in raw_dialogue:
                if isinstance(turn, dict):
                    turns.append({
                        'speaker': turn.get('speaker', turn.get('role', 'Seeker')),
                        'text': turn.get('text', turn.get('content', turn.get('utterance', ''))),
                    })
        target_text = str(item.get('current_text', item.get('target', '')))
        label = None
        if has_labels:
            raw_label = item.get('label', item.get('defense_level'))
            if raw_label is not None:
                label = int(raw_label)
        samples.append(DialogueSample(
            sample_id=sample_id, dialogue_id=dialogue_id,
            turns=turns, target_text=target_text, label=label,
        ))

    print(f'Loaded {len(samples)} samples (labels={has_labels})')
    return samples


class DefenseDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=512, max_turns=40):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_turns = max_turns

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        text = format_dialogue(s.turns, s.target_text, self.max_turns, s.target_turn_index)
        enc = self.tokenizer(text, max_length=self.max_length, padding='max_length',
                             truncation=True, return_tensors='pt')
        item = {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }
        if s.label is not None:
            item['labels'] = torch.tensor(s.label, dtype=torch.long)
        return item

    def get_labels(self):
        return [s.label for s in self.samples if s.label is not None]


def create_fold_datasets(samples, tokenizer, num_folds=5, fold_index=0,
                         max_length=512, max_turns=40):
    dialogue_ids = np.array([s.dialogue_id for s in samples])
    labels = np.array([s.label if s.label is not None else 0 for s in samples])
    gkf = GroupKFold(n_splits=num_folds)
    splits = list(gkf.split(range(len(samples)), labels, dialogue_ids))
    train_idx, val_idx = splits[fold_index]

    train_samples = [samples[i] for i in train_idx]
    val_samples = [samples[i] for i in val_idx]

    # Verify no leakage
    train_ids = {s.dialogue_id for s in train_samples}
    val_ids = {s.dialogue_id for s in val_samples}
    assert len(train_ids & val_ids) == 0, 'Dialogue leakage!'

    print(f'Fold {fold_index+1}/{num_folds}: train={len(train_samples)}, val={len(val_samples)}')
    return (
        DefenseDataset(train_samples, tokenizer, max_length, max_turns),
        DefenseDataset(val_samples, tokenizer, max_length, max_turns),
    )

print('Dataset module ready')

In [ ]:
# Cell 6: Model (Focal Loss + R-Drop + DeBERTa classifier)

import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        if alpha is not None:
            self.register_buffer('alpha', alpha.float())
        else:
            self.alpha = None

    def forward(self, inputs, targets):
        num_classes = inputs.size(-1)
        log_probs = F.log_softmax(inputs, dim=-1)
        probs = torch.exp(log_probs)

        if self.label_smoothing > 0:
            smooth = self.label_smoothing / num_classes
            one_hot = torch.zeros_like(log_probs).scatter(1, targets.unsqueeze(1), 1.0)
            one_hot = one_hot * (1.0 - self.label_smoothing) + smooth
            loss = -(one_hot * log_probs).sum(dim=-1)
            pt = (one_hot * probs).sum(dim=-1)
        else:
            loss = F.nll_loss(log_probs, targets, reduction='none')
            pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)

        focal_weight = (1.0 - pt) ** self.gamma
        loss = focal_weight * loss

        if self.alpha is not None:
            at = self.alpha.to(inputs.device).gather(0, targets)
            loss = at * loss

        return loss.mean()


def compute_r_drop_loss(logits_1, logits_2, reduction='batchmean'):
    p = F.log_softmax(logits_1, dim=-1)
    q = F.log_softmax(logits_2, dim=-1)
    kl_pq = F.kl_div(p, q.exp(), reduction=reduction)
    kl_qp = F.kl_div(q, p.exp(), reduction=reduction)
    return (kl_pq + kl_qp) / 2.0


class DefenseClassifier(nn.Module):
    def __init__(self, model_name='microsoft/deberta-v3-base', num_labels=9,
                 class_weights=None, focal_gamma=2.0, label_smoothing=0.05,
                 r_drop_lambda=0.5, r_drop_enabled=True):
        super().__init__()
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=num_labels)
        self.r_drop_lambda = r_drop_lambda
        self.r_drop_enabled = r_drop_enabled
        self.criterion = FocalLoss(alpha=class_weights, gamma=focal_gamma,
                                   label_smoothing=label_smoothing)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs_1 = self.model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs_1.logits
        result = {'logits': logits}

        if labels is not None:
            cls_loss = self.criterion(logits, labels)
            if self.r_drop_enabled and self.training:
                outputs_2 = self.model(input_ids=input_ids, attention_mask=attention_mask)
                cls_loss = (cls_loss + self.criterion(outputs_2.logits, labels)) / 2.0
                r_drop_loss = compute_r_drop_loss(logits, outputs_2.logits)
                result['loss'] = cls_loss + self.r_drop_lambda * r_drop_loss
            else:
                result['loss'] = cls_loss

        return result

print('Model module ready')

In [ ]:
# Cell 7: Load data and inspect distribution

from collections import Counter

train_samples = load_psydefconv('data/train.json', has_labels=True)
test_samples = load_psydefconv('data/test.json', has_labels=False)

label_counts = Counter(s.label for s in train_samples)
print('\nLabel Distribution:')
for lbl in sorted(label_counts):
    pct = 100 * label_counts[lbl] / len(train_samples)
    print(f'  {lbl} ({DEFENSE_LABELS[lbl]:25s}): {label_counts[lbl]:4d} ({pct:.1f}%)')

In [ ]:
# Cell 8: Training loop

import time
from sklearn.metrics import classification_report, f1_score
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

# --- Config ---
MODEL_NAME = 'microsoft/deberta-v3-base'
NUM_FOLDS = 5
NUM_EPOCHS = 6
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM = 2
LR = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_LENGTH = 512
MAX_TURNS = 40
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.05
R_DROP_LAMBDA = 0.5
OUTPUT_DIR = 'checkpoints'
# --- End Config ---

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


@torch.no_grad()
def evaluate(model, val_loader):
    model.eval()
    all_preds, all_labels = [], []
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids, attention_mask)
        preds = outputs['logits'].argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(batch['labels'].numpy().tolist())
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    return macro_f1, weighted_f1, all_preds, all_labels


def train_fold(fold_index):
    print(f'\n{"="*60}')
    print(f'FOLD {fold_index + 1}/{NUM_FOLDS}')
    print(f'{"="*60}')

    train_ds, val_ds = create_fold_datasets(
        train_samples, tokenizer, NUM_FOLDS, fold_index, MAX_LENGTH, MAX_TURNS)

    class_weights = compute_class_weights(train_ds.get_labels())

    model = DefenseClassifier(
        model_name=MODEL_NAME, num_labels=NUM_LABELS,
        class_weights=class_weights, focal_gamma=FOCAL_GAMMA,
        label_smoothing=LABEL_SMOOTHING, r_drop_lambda=R_DROP_LAMBDA,
    ).to(device)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)

    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    optimizer_grouped = [
        {'params': [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay) and p.requires_grad],
         'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay) and p.requires_grad],
         'weight_decay': 0.0},
    ]

    total_steps = len(train_loader) // GRAD_ACCUM * NUM_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)

    optimizer = torch.optim.AdamW(optimizer_grouped, lr=LR, eps=1e-8)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

    best_f1 = 0.0
    fold_dir = f'{OUTPUT_DIR}/fold_{fold_index}'
    os.makedirs(fold_dir, exist_ok=True)

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_loss = 0.0
        t0 = time.time()

        for step, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            if scaler:
                with torch.amp.autocast('cuda'):
                    outputs = model(input_ids, attention_mask, labels)
                    loss = outputs['loss'] / GRAD_ACCUM
                scaler.scale(loss).backward()
            else:
                outputs = model(input_ids, attention_mask, labels)
                loss = outputs['loss'] / GRAD_ACCUM
                loss.backward()

            epoch_loss += loss.item() * GRAD_ACCUM

            if (step + 1) % GRAD_ACCUM == 0:
                if scaler:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        elapsed = time.time() - t0
        avg_loss = epoch_loss / max(len(train_loader), 1)

        macro_f1, weighted_f1, _, _ = evaluate(model, val_loader)
        print(f'Epoch {epoch+1}/{NUM_EPOCHS} | '
              f'Loss: {avg_loss:.4f} | '
              f'Val macro-F1: {macro_f1:.4f} | '
              f'Val weighted-F1: {weighted_f1:.4f} | '
              f'{elapsed:.0f}s')

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            torch.save({
                'model_state_dict': model.state_dict(),
                'fold_index': fold_index,
                'epoch': epoch + 1,
                'macro_f1': best_f1,
                'config': {
                    'base_model': MODEL_NAME,
                    'num_labels': NUM_LABELS,
                },
            }, f'{fold_dir}/best_model.pt')
            print(f'  >> New best model saved (macro-F1={best_f1:.4f})')

    # Final report for this fold
    macro_f1, weighted_f1, preds, labels_gt = evaluate(model, val_loader)
    label_names = [DEFENSE_LABELS[i] for i in range(NUM_LABELS)]
    print(f'\nFold {fold_index+1} Final Report (best macro-F1={best_f1:.4f}):')
    print(classification_report(labels_gt, preds, target_names=label_names, zero_division=0))

    del model
    torch.cuda.empty_cache()
    return best_f1

print('Training function ready')

In [ ]:
# Cell 9: Run all 5 folds

all_f1s = []
total_start = time.time()

for fold_idx in range(NUM_FOLDS):
    f1 = train_fold(fold_idx)
    all_f1s.append(f1)

total_elapsed = time.time() - total_start
print(f'\n{"="*60}')
print(f'TRAINING COMPLETE')
print(f'{"="*60}')
print(f'Average macro-F1: {np.mean(all_f1s):.4f} (±{np.std(all_f1s):.4f})')
for i, f1 in enumerate(all_f1s):
    print(f'  Fold {i+1}: {f1:.4f}')
print(f'Total time: {total_elapsed/60:.1f} minutes')

In [ ]:
# Cell 11: Download results
import shutil
from google.colab import files

# Download submission
files.download('submission.jsonl')

# Download checkpoints as a zip
shutil.make_archive('checkpoints', 'zip', 'checkpoints')
files.download('checkpoints.zip')

print('Downloads triggered')

In [ ]:
# Cell 11: Download results

from google.colab import files

# Download submission
files.download('submission.jsonl')

# Download checkpoints as a zip
!zip -r checkpoints.zip checkpoints/
files.download('checkpoints.zip')

print('Downloads triggered')